# Fire & Smoke Detection — YOLOv8 on D-Fire (Drive-persistent)

Self-contained Colab notebook that:
1. Downloads the **D-Fire** dataset (~21k images, fire+smoke YOLO annotations)
2. Normalizes the directory layout to the Ultralytics-expected structure
3. **Mounts Google Drive** so training runs survive Colab disconnects
4. Trains **YOLOv8s** for 30 epochs on a free T4 GPU
5. Evaluates on the test split (mAP@0.5, mAP@0.5:0.95, precision, recall)
6. Runs inference on user-provided sample fire photos
7. Packages the trained `best.pt` for download

**Runtime needed:** Free T4 GPU (Runtime → Change runtime type → T4 GPU).
**Time estimate:** ~60–80 minutes end-to-end.

Dataset source: [gaiasd/DFireDataset](https://github.com/gaiasd/DFireDataset) (~21k images, fire + smoke bbox annotations).
Mirror: [Kaggle/shubhamkarande13/d-fire](https://www.kaggle.com/datasets/shubhamkarande13/d-fire).

> **Why Drive-save?** A previous run lost the trained model when the Colab VM was reset before the packaging cell ran. By writing training runs directly to `/content/drive/MyDrive/fire_runs/`, every checkpoint persists across disconnects — if the runtime dies, you just remount Drive on a new session and the weights are still there.

## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics==8.3.40 kagglehub Pillow

## 1.5 Mount Google Drive (NEW — persists training runs)

The first time you run this, Colab pops up a permissions dialog. Click "Connect to Google Drive" and allow access. Subsequent runs reuse the saved auth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_RUNS = '/content/drive/MyDrive/fire_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('Training runs will be saved to:', DRIVE_RUNS)

## 2. Download D-Fire

Two paths — the notebook tries them in order:
1. **Kaggle mirror** via `kagglehub` (fastest, no auth file needed for public datasets in Colab).
2. **GitHub release** fallback if Kaggle is unavailable.

If `kagglehub` prompts for a Kaggle API token: go to https://www.kaggle.com/settings/account → "Create New Token" → upload `kaggle.json` to `/root/.config/kaggle/`.

In [ ]:
import os
import shutil
from pathlib import Path

DATA_ROOT = Path('/content/dfire_raw')
DATA_ROOT.mkdir(exist_ok=True)

try:
    import kagglehub
    dl_path = kagglehub.dataset_download('shubhamkarande13/d-fire')
    print('kagglehub downloaded to:', dl_path)
    DATA_ROOT = Path(dl_path)
except Exception as e:
    print('kagglehub failed:', e)
    print('Falling back to git clone of the official repo (note: this only grabs sample data, not the full 21k).')
    !git clone --depth=1 https://github.com/gaiasd/DFireDataset.git /content/DFireDataset
    DATA_ROOT = Path('/content/DFireDataset')

In [ ]:
# Inspect what we got
for p in sorted(DATA_ROOT.rglob('*'))[:50]:
    print(p.relative_to(DATA_ROOT))
print('...')
print('Total files:', sum(1 for _ in DATA_ROOT.rglob('*') if _.is_file()))

## 3. Normalize to Ultralytics YOLO layout

Target structure:
```
/content/dfire/
  images/{train,val,test}/*.jpg
  labels/{train,val,test}/*.txt
  data.yaml
```

Classes: `0 = fire`, `1 = smoke` (per D-Fire convention).

In [ ]:
import random
import yaml

DST = Path('/content/dfire')
for sub in ['images/train', 'images/val', 'images/test', 'labels/train', 'labels/val', 'labels/test']:
    (DST / sub).mkdir(parents=True, exist_ok=True)

# Find all images and matching labels
exts = {'.jpg', '.jpeg', '.png'}
all_images = []
for img in DATA_ROOT.rglob('*'):
    if img.suffix.lower() in exts and 'images' in img.parts:
        candidates = [
            img.parent.parent / 'labels' / (img.stem + '.txt'),
            Path(str(img.parent).replace('images', 'labels')) / (img.stem + '.txt'),
        ]
        label = next((c for c in candidates if c.exists()), None)
        if label is not None:
            all_images.append((img, label))

print(f'Found {len(all_images)} image/label pairs.')
assert len(all_images) > 1000, 'Too few pairs — dataset structure may have changed; inspect DATA_ROOT.'

# Detect if dataset already has train/val/test split
split_in_path = {'train': [], 'val': [], 'test': []}
for img, lbl in all_images:
    parts_lower = [p.lower() for p in img.parts]
    if 'train' in parts_lower:
        split_in_path['train'].append((img, lbl))
    elif any(s in parts_lower for s in ['val', 'valid', 'validation']):
        split_in_path['val'].append((img, lbl))
    elif 'test' in parts_lower:
        split_in_path['test'].append((img, lbl))

use_existing_split = all(len(v) > 100 for v in split_in_path.values())

if use_existing_split:
    print('Using existing train/val/test split from source.')
    splits = split_in_path
else:
    print('No usable split found — creating a fresh 80/10/10 split.')
    random.seed(42)
    random.shuffle(all_images)
    n = len(all_images)
    splits = {
        'train': all_images[: int(0.8 * n)],
        'val':   all_images[int(0.8 * n): int(0.9 * n)],
        'test':  all_images[int(0.9 * n):],
    }

for split_name, pairs in splits.items():
    for img, lbl in pairs:
        shutil.copy(img, DST / 'images' / split_name / img.name)
        shutil.copy(lbl, DST / 'labels' / split_name / lbl.name)
    print(f'{split_name}: {len(pairs)} pairs')

data_yaml = {
    'path': str(DST),
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc': 2,
    'names': ['fire', 'smoke'],
}
with open(DST / 'data.yaml', 'w') as f:
    yaml.safe_dump(data_yaml, f)
print('\nWrote', DST / 'data.yaml')
print(open(DST / 'data.yaml').read())

### Sanity check: visualize a few samples with their boxes

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

NAMES = ['fire', 'smoke']
COLORS = {0: 'red', 1: 'gray'}

def draw_yolo(img_path, lbl_path):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    draw = ImageDraw.Draw(img)
    with open(lbl_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            c, x, y, w, h = int(parts[0]), *map(float, parts[1:5])
            x1, y1 = (x - w / 2) * W, (y - h / 2) * H
            x2, y2 = (x + w / 2) * W, (y + h / 2) * H
            draw.rectangle([x1, y1, x2, y2], outline=COLORS[c], width=3)
            draw.text((x1, max(0, y1 - 12)), NAMES[c], fill=COLORS[c])
    return img

sample_imgs = list((DST / 'images/train').glob('*'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, p in zip(axes.flat, sample_imgs):
    lbl = DST / 'labels/train' / (p.stem + '.txt')
    ax.imshow(draw_yolo(p, lbl))
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Train YOLOv8s (30 epochs, runs saved to Drive)

**Why 30 epochs?** D-Fire on YOLOv8s typically plateaus around epoch 30–35. With `patience=10` it'll early-stop if val mAP stops improving. 50 epochs only buys ~1–3 mAP points over 30, and the extra ~30 min on a free T4 increases the chance of a disconnect.

**Hyperparameters:**
- `imgsz=640` — standard, balances detail vs throughput
- `epochs=30` — D-Fire plateaus around 30–35; early-stop catches the rest
- `patience=10` — stop if no val improvement for 10 epochs
- `batch=32` — fits T4 (16 GB)
- `close_mosaic=10` — disable mosaic for the last 10 epochs
- `project=DRIVE_RUNS` — checkpoints land on Drive, not the ephemeral VM

If the runtime dies mid-training, re-mount Drive and rerun this cell with `resume=True` added — Ultralytics picks up from the last Drive checkpoint.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # COCO-pretrained starting weights

results = model.train(
    data=str(DST / 'data.yaml'),
    epochs=30,
    imgsz=640,
    batch=32,
    patience=10,
    close_mosaic=10,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    project=DRIVE_RUNS,
    name='fire_yolov8s',
    device=0,
    workers=2,
    seed=42,
    verbose=True,
)

## 5. Evaluate on test split

In [ ]:
best_pt = Path(DRIVE_RUNS) / 'fire_yolov8s' / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt missing at {best_pt}'

model = YOLO(str(best_pt))
metrics = model.val(data=str(DST / 'data.yaml'), split='test', imgsz=640, plots=True)
print('mAP@0.5      :', round(float(metrics.box.map50), 4))
print('mAP@0.5:0.95 :', round(float(metrics.box.map), 4))
print('Precision    :', round(float(metrics.box.mp), 4))
print('Recall       :', round(float(metrics.box.mr), 4))

## 6. Sanity-check on real fire photos

Upload a few photos (your own, or grab some via `wget`) to `/content/test_fire/`. The cell below runs inference and shows boxes + confidences.

In [ ]:
TEST_DIR = Path('/content/test_fire')
TEST_DIR.mkdir(exist_ok=True)

# Quick smoke-test grabs (open-licensed Wikimedia images of fire)
samples = [
    'https://upload.wikimedia.org/wikipedia/commons/thumb/3/36/Large_bonfire.jpg/1024px-Large_bonfire.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/a/aa/Caldor_Fire_Plume_22.jpg/1024px-Caldor_Fire_Plume_22.jpg',
    'https://upload.wikimedia.org/wikipedia/commons/thumb/2/24/Forest_fire_in_Greece%2C_August_2009.jpg/1024px-Forest_fire_in_Greece%2C_August_2009.jpg',
]
for i, url in enumerate(samples):
    out = TEST_DIR / f'sample_{i}.jpg'
    if not out.exists():
        !wget -q -O {out} {url}

results = model.predict(source=str(TEST_DIR), imgsz=640, conf=0.25, save=True)
for r in results:
    print(r.path, '→', len(r.boxes), 'detections')
    for b in r.boxes:
        print(f'  {NAMES[int(b.cls)]} conf={float(b.conf):.2f}')

# Show the saved annotated images
import glob
from IPython.display import Image as IPyImage, display
for p in sorted(glob.glob(str(Path(results[0].save_dir) / '*.jpg'))):
    print(p)
    display(IPyImage(p))

## 7. Package the trained model for local use

The weights are **already on Drive** at `MyDrive/fire_runs/fire_yolov8s/weights/best.pt`. This cell just packages a clean release zip with the model + training plots for direct browser download. If this cell fails for any reason, the weights are still on Drive — you can grab them from the Drive web UI directly.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

src = Path(DRIVE_RUNS) / 'fire_yolov8s'
out_dir = Path('/content/fire_model_release')
out_dir.mkdir(exist_ok=True)
shutil.copy(src / 'weights/best.pt', out_dir / 'fire.pt')
for f in ['results.png', 'results.csv', 'confusion_matrix.png', 'PR_curve.png']:
    if (src / f).exists():
        shutil.copy(src / f, out_dir / f)

shutil.make_archive('/content/fire_model_release', 'zip', out_dir)

# Belt-and-braces: also stash the zip on Drive
shutil.copy('/content/fire_model_release.zip', '/content/drive/MyDrive/fire_model_release.zip')
print('Saved zip to Drive: MyDrive/fire_model_release.zip')

files.download('/content/fire_model_release.zip')

## What you should expect

| Metric | Expected range (D-Fire test, 30 epochs) |
| --- | --- |
| mAP@0.5 | 0.75 – 0.82 |
| mAP@0.5:0.95 | 0.45 – 0.52 |
| Precision (fire) | 0.80 – 0.88 |
| Recall (smoke) | 0.65 – 0.78 |

If recall on smoke is below 0.6, resume training with `resume=True` for another 10–20 epochs (Ultralytics picks up from the Drive checkpoint automatically).

## Recovery if the runtime dies

On a fresh runtime:
1. Run cells 0–6 (install + Drive mount).
2. The weights are still at `/content/drive/MyDrive/fire_runs/fire_yolov8s/weights/`.
3. Skip dataset cells (4–9), skip training (15), jump to eval (17) or packaging (21).

To *continue* training where it stopped, add `resume=True` to the train call and rerun — Ultralytics finds the latest Drive checkpoint and resumes.

## Next steps after this notebook

1. Unzip `fire_model_release.zip` and drop `fire.pt` into the repo at `model/fire.pt`.
2. Run `python model/predict_fire.py path/to/photo.jpg` locally for a quick test.
3. Wire the `/fire_analysis` endpoint into `app.py`.